# Step 1: Category Data Preparation

This notebook ingests categorization data for YouTube videos from folders specified in "config/data_config.yml", returning a series of files containing
1. a cleaned, deduplicated set of categorizations for all videos if available ("combined_cleaned_data.csv")
2. a list of validation mismatches if any ("validation_mismatches")
3. a list of remaining duplicates
4. a normalization dictionary "raw_dict.csv" that can be edited and will be integrated in the process to normalize terms that are misspelled if saved under the filename "ytma_normalized_terms_dict.csv"

In [ ]:
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt
from datetime import datetime
from pandas import ExcelWriter
import numpy as np
import yaml
import pprint

In [ ]:
%pip install yt-dlp
import yt_dlp

In [ ]:
# flags

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
output = './output/'
print(os.listdir(output))

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}

pprint.pprint(directories)

## Data Ingest

In [ ]:
expected_cat_dict = {
    'Main Video Source (choose 1)': [
        'Animation', 
        'Game', 
        'Live-action Performance', 
        'Mixed'], 
    'Formal Elements (multiple possible)': [
        'Avatar', 'Camera', 'Text'],
    'Other Significant Tags (please describe new tags in the tag description column)': [

    ],  # missing comma fixed here
    'YouTube Shorts (yes no)': ['Yes', 'No'],
    'Other Noteworthy Formal Elements': [
        
    ],
    'Content Type (choose 0-1)': [
        'Creative',
        'Experiment',
        'Game Comparison',
        'Reaction Video',
        'Speedrunning',
        'Tournament',
        'Training',
        'Walkthrough',
        'Opinion/Review'
    ],
    'Content Focus (choose 0-2)': [
        'Boss fight',
        'Character',
        'Crafting',
        'Derivative',
        'Equipment',
        'Glitch',
        'Graphics',
        'MODs',
        'OST/Music',
        'PvP mode',
        'Story'
    ],
    'Other Noteworthy Content Elements': [],
    'Values (Subjective Evaluation) muliple possible': [
        'Comedic',
        'Community-building',
        'Entertaining',
        'Informative',
        'Promotional'
    ],
    'Other Noteworthy Evaluation': [],
    'new tags and other memos': []
}


# Define folder and language tags
lang_tags = ['jp', 'ko', 'hant', 'hans', 'en']

languagecodestoreplace = {'ko': 'ko', 
                          'en': 'en', 
                          'hant': 'zh-Hant',
                          'zh-Hant': 'zh-Hant', 
                          'jp': 'ja', 
                          'ja': 'ja', 
                          'hans': 'zh-Hans',
                          'zh-Hans': 'zh-Hans'}

all_terms = []


# Target columns (original casing)
target_columns = [
    "videoId", "link", "Content match (yes no)", "Language Match (yes no)",
    "Main Video Source (choose 1)", "Formal Elements (multiple possible)",
    "Other Significant Tags (please describe new tags in the tag description column)",
    "YouTube Shorts (yes no)", "Other Noteworthy Formal Elements",
    "Content Type (choose 0-1)", "Content Focus (choose 0-2)",
    "Other Noteworthy Content Elements", "Values (Subjective Evaluation) muliple possible",
    "Other Noteworthy Evaluation", "new tags and other memos"
]

# Target columns (original casing)
standardized_columns = [
    "Content match (yes no)", "Language Match (yes no)",
    "Main Video Source (choose 1)", "Formal Elements (multiple possible)", 
    "Other Significant Tags (please describe new tags in the tag description column)",
    "YouTube Shorts (yes no)", "Other Noteworthy Formal Elements", "Content Type (choose 0-1)", "Content Focus (choose 0-2)",
    "Values (Subjective Evaluation) muliple possible",
]

# Create lowercase mapping for columns
target_columns_lower = [col.lower() for col in target_columns]
column_rename_map = dict(zip(target_columns_lower, target_columns))  # lowercase -> original
standardized_columns_lower = [col.lower() for col in standardized_columns]

In [ ]:
def extract_lang_tag(filename):
    filename_lower = filename.lower()
    for tag in lang_tags:
        if tag in filename_lower:
            return languagecodestoreplace[tag]
    return 'unknown'


def createcatdata(folder, game_label):
    result = {}
    result['all_files'] = glob(os.path.join(folder, '*.xlsx'))
    result['combined_data'] = []
    result['incomplete_rows'] = []

    # combine all data
    for file in result['all_files']:
        base_filename = os.path.basename(file)
        lang = extract_lang_tag(base_filename)
        print(f"\n📄 Processing: {base_filename}  |  Language tag: {lang}")
        try:
            df = pd.read_excel(file, dtype=str, engine="openpyxl")
            if df.empty:
                print("⚠️ File is empty!")
                continue
            # Normalize column names
            df.columns = df.columns.str.strip().str.lower()

            # Check for overlap with expected columns
            available_cols = [col for col in df.columns if col in target_columns_lower]
            if not available_cols:
                print("⚠️ No matching columns. Skipping.")
                continue

            # Keep only relevant columns, rename them to original casing
            df = df[available_cols].rename(columns=column_rename_map)

            df['videoSearchRegion'] = lang
            df['source_file'] = base_filename
            if 'link' in df.columns:
                df['link'] = df['link'].astype(str)
                df['videoId'] = df['link'].apply(lambda x: x.split('v=')[1] if 'v=' in x else None)
            else:
                print('❌no link in file: ', file)
            df['videoId'] = df['videoId'].str.strip()
            valid_mask = df['videoId'].notna() & df['videoId'] != ''
            valid_df = df[valid_mask].copy()
            invalid_df = df[~valid_mask].copy()

            if len(invalid_df) > 0:
                print(f"✅ Valid rows: {len(valid_df)}  |  🚫 Incomplete rows: {len(invalid_df)}")

            result['combined_data'].append(valid_df)
            if not invalid_df.empty:
                result['incomplete_rows'].append(invalid_df)
        except Exception as e:
            print(f"❌ Error reading {file}: {e}")
        
    # deduplication, filtering, normalization

    combined_df = pd.concat(result['combined_data'], ignore_index=True)
    print(f"\n📊 Total combined rows before deduplication: {combined_df}")
    # Filter where Language Match is not 'no'
    filtered_df = combined_df[combined_df['Language Match (yes no)'].str.strip().str.lower() != 'no'].copy()
    print(f"Rows with Language Match != 'no': {len(filtered_df)}")
    # handle normalization dict
    new_dict = {}
    try:
        tmp = pd.read_csv(os.path.join(folder, 'ytma_normalized_terms_dict.csv'))
        normalized_dict = dict(zip(tmp['key'], tmp['value']))
        print(normalized_dict)
    except:
        print('terms normalization dict could not be loaded')
        normalized_dict = False

    # Normalize string columns for comparison
    for col in [x for x in filtered_df.columns if x not in ('link', 'videoId')]:
        if filtered_df[col].dtype == object:
            filtered_df[col] = filtered_df[col].fillna('')
            
            def fixnormalizedterms(t):
                if normalized_dict:
                    if t in normalized_dict.keys():
                        #print('turned: ', t, "to: ", normalized_dict[t])
                        return str(normalized_dict[t])
                else:
                    print('no norm dict available')
                return t

            def normalize_and_join(cell):
                cell = cell.lower().replace(' ','').replace('；',';').replace('\n', ';').replace('\r', '').replace('?', '').strip().rstrip(';；')
                terms = [term for term in cell.split(';') if term]
                #print(terms)
                terms = [fixnormalizedterms(term) for term in terms]
                #print(terms)
                # Add terms to global list for frequency counting
                all_terms.extend(terms)
                # Join cleaned terms back into a string
                return '; '.join(sorted(terms))
            
            filtered_df[col] = filtered_df[col].apply(normalize_and_join)
        
        if col in standardized_columns:
            tmpdf_exploded = filtered_df[col].str.split(';').explode()
            unique_values = tmpdf_exploded.unique()
            tmp = {x: x for x in unique_values}
            new_dict.update(tmp)

            #add all terms toconsider when looking for duplicates (ignore 'source_file')
    dedup_columns = [col for col in filtered_df.columns if col != 'source_file']

    new_dict_df = pd.DataFrame.from_dict(new_dict, orient='index')
    new_dict_df.to_csv(os.path.join(folder, 'ytma_raw_dict.csv'))

    # Sort by videoId and source_file (so we can keep earliest or latest)
    filtered_df = filtered_df.sort_values(by=['videoId', 'source_file'], ascending=[True, True])

    # Define which columns to consider when looking for duplicates (ignore 'source_file')
    dedup_columns = [col for col in filtered_df.columns if col != 'source_file']

    # Drop duplicates based on content across files (ignore source_file)
    #deduped_df = filtered_df.drop_duplicates(subset=dedup_columns, keep='first')
    #print(deduped_df.columns)

    before_dedup = len(filtered_df)
    deduped_df = filtered_df.drop_duplicates(subset=dedup_columns, keep='first')
    after_dedup = len(deduped_df)
    print(f"🧹 Removed {before_dedup - after_dedup} duplicates based on content ignoring source_file.")

    #count the content in each row, excluding videoid and link
    deduped_df['content_count'] = deduped_df.drop(columns=['videoId', 'link', 'videoSearchRegion', 'source_file']).replace('', np.nan).notna().sum(axis=1)
    
    deduped_df = deduped_df[deduped_df['content_count'] > 0]
    after_emptyremoval = len(deduped_df)
    print(f"🧹 Removed {after_dedup - after_emptyremoval} duplicates based on empty content.")

    duplicates_remaining = deduped_df.duplicated(subset=dedup_columns, keep=False)
    print("Remaining content duplicates:", duplicates_remaining.sum())

    # integrate clean data
    result['clean_data'] = deduped_df

    # Validate columns against expected categories

    def normalize_term(term):
        return term.lower().replace(' ', '').strip().rstrip(';；')

    # Create a normalized version of the expected_cat_dict
    normalized_expected_cat_dict = {
        k: sorted({normalize_term(v) for v in values})
        for k, values in expected_cat_dict.items()
        if values  # only include keys with defined lists
    }
    validation_results = []

    for col, allowed_values in normalized_expected_cat_dict.items():
        if col not in deduped_df.columns:
            print(f"⚠️ Column '{col}' not found in data.")
            continue

        print(f"🔍 Validating values in: '{col}'")

        for idx, row in deduped_df.iterrows():
            raw_terms = row[col]
            terms = raw_terms.split('; ') if raw_terms else []

            for term in terms:
                normalized_term = normalize_term(term)
                if normalized_term not in allowed_values:
                    validation_results.append({
                        'videoId': row.get('videoId'),
                        'column': col,
                        'invalid_term': term,
                        'normalized_term': normalized_term,
                        'source_file': row.get('source_file'),
                        'lang': row.get('lang')
                    })
    # Save mismatches (if any)
    if validation_results:
        invalid_df = pd.DataFrame(validation_results)
        invalid_df.to_csv(os.path.join(folder, f'ytma_{game_label}_validation_mismatches.csv'), index=False)
        print(f"❌ Found {len(invalid_df)} invalid entries. Saved to 'validation_mismatches.csv'")
    else:
        print("✅ All validated terms matched expected values.")
    
    # Save cleaned combined data
    deduped_df.to_csv(os.path.join(folder, f'ytma_{game_label}_combined_cleaned_data.csv'), index=False)
    print(f"✅ Final cleaned data saved to 'combined_cleaned_data.csv' with {len(deduped_df)} rows.")

    # Optionally save duplicate ids still found based on video id
    duplicate_videoid_mask = deduped_df.duplicated(subset=['videoId'], keep=False)
    remaining_id_duplicates = deduped_df[duplicate_videoid_mask].sort_values("videoId")

    if not remaining_id_duplicates.empty:
        remaining_id_duplicates.to_csv(os.path.join(folder, f'ytma_{game_label}_remaining_id_duplicates.csv'), index=False)
        print(f"📝 Saved {len(remaining_id_duplicates)} duplicate videoId rows to 'remaining_id_duplicates.csv'")

    # Save incomplete rows
    if result['incomplete_rows']:
        incomplete_df = pd.concat(result['incomplete_rows'], ignore_index=True)
        incomplete_df.to_csv(os.path.join(folder, f'ytma_{game_label}_incomplete_rows.csv'), index=False)
        print(f"📝 Incomplete rows saved to 'incomplete_rows.csv' | Total: {len(incomplete_df)} rows")

    return(result['clean_data'])

In [ ]:
# set true if you want all category data to be merged and deduplicated and overwrite existing cat data
forcecatfilecreation = True

# set true if you want to test if the video still exists. Requires YouTube API credentials
testavailability = False

# create all neccessary data
datadict = {}

for game in directories.keys():
    # Load all Excel files
    print(f'processing: {game}')
    datadict[game] = {}
    datadict[game]['catdir'] = directories[game]['catdir']
    tmpcatf =  glob(os.path.join(datadict[game]['catdir'], catfile))
    if tmpcatf:
        datadict[game]['catfile'] = tmpcatf[0]
        print(f'catfile detected: {datadict[game]['catfile']}')
        datadict[game]['catdata'] = pd.read_csv(datadict[game]['catfile'])
    else:
        print('no category file exists')
        datadict[game]['catfile'] = False
    if forcecatfilecreation or (datadict[game]['catfile']==False):
        print ('creating category data')
        datadict[game]['catdata'] = createcatdata(datadict[game]['catdir'], game)  
    # add availability test version 2, less calls to the web (hopefully)
    # run only once final version is ready
    if testavailability: 
        print(datadict[game]['catdata'].columns)
        verified_video_df = datadict[game]['catdata'][['videoId', 'link']].copy().drop_duplicates(subset=['videoId'], keep='first')
        print(len(verified_video_df))

        # this will get restrictions on YouTube? NO!
        def check_video_exists(url):
            try:
                with yt_dlp.YoutubeDL({'quiet': True}) as ydl:
                    ydl.extract_info(url, download=False)
                return "Exists"
            except yt_dlp.utils.DownloadError as e:
                return str(e).split('\n')[0]  # First line of the error message
            except Exception as e:
                return f"Failed: {type(e).__name__}"

        # Perform the checks and add status
        verified_video_df['video_status'] = verified_video_df['link'].apply(check_video_exists)

        # Add the date of check (same date for all rows here)
        verified_video_df['checked_date'] = datetime.now().date()

        merged_dataframe = deduped_df.merge(verified_video_df, on='videoId', how='left')
        merged_dataframe.to_csv(os.path.join(folder, f'ytma_{game_label}_verified_combined_cleaned_data.csv'), index=False)

        datadict[game]['verified_catdata'] = merged_dataframe